In [3]:
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import os


Define parameters


In [80]:
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io
import sys
import os


def convert_utc_to_local(df, local_tz):
    """
    Convert the datetime index of the DataFrame from UTC to a local timezone.

    Args:
    df : pandas.DataFrame
        DataFrame with a datetime index in UTC.
    local_tz : str
        A timezone string (e.g., 'America/Chicago').

    Returns:
    pandas.DataFrame
        DataFrame with datetime index converted to the specified local timezone.
    """
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df = df.reset_index(level='station', drop=True)
        df.index = pd.to_datetime(df.index)

    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    
    df.index = df.index.tz_convert(local_tz)
    return df


def filter_dataframe_by_date(df, start_date, end_date, timezone=None):
    """
    Filter the DataFrame to include rows between the specified start and end dates,
    handling timezone differences appropriately.
    """
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    if timezone:
        start_date = start_date.tz_localize(timezone)
        end_date = end_date.tz_localize(timezone)
    else:
        df.index = df.index.tz_localize(None)

    return df.loc[(df.index >= start_date) & (df.index <= end_date)]


def get_parameters_MERRA2(lat, lon, year):
    api_endpoint = f"https://power.larc.nasa.gov/api/temporal/hourly/point?community=SB&parameters=&longitude={lon}&latitude={lat}&start={year}0101&end={year}1231&format=EPW"
    response = requests.get(api_endpoint)
    csv_data = io.StringIO(response.text)
    df = pd.read_csv(csv_data, skiprows=8, header=None)
    header = '\n'.join(response.text.splitlines()[:8])
    return df, header


def merge_data(df, data):
    """
    Merges two datasets, replacing specific columns in `df` with corresponding values from `data`.
    """
    for col in [6, 7, 8, 33, 30, 21, 20, 9]:  # Replace indices with more descriptive names if possible
        if not df[col].isna().all():
            df[col] = list(data['temp'][1:])
    return df


def check_missing_hours(year, df):
    """
    Checks for missing hours in the DataFrame's datetime index for a specified year.
    """
    full_index = pd.date_range(start=f"{year}-01-01", end=f"{year+1}-01-01", freq="H")
    missing_hours = full_index.difference(df.index)
    missing_hours_num = len(missing_hours)

    if missing_hours_num > 0:
        diffs = missing_hours.to_series().diff().dt.total_seconds().div(3600)
        largest_consecutive_group = (diffs != 1).cumsum().value_counts().max()
    else:
        largest_consecutive_group = 0

    return missing_hours_num, largest_consecutive_group


def fix_wmo(wmo):
    """
    Attempts to fix or standardize the WMO code format.
    """
    if wmo:
        try:
            return str(int(wmo))
        except ValueError:
            icao = wmo
            return get_wmo_from_icao_NOAA(icao) or icao
    return wmo


def get_noaa_merra2_data(lat, lon, year, file_type, save_folder):
    """
    Retrieves NOAA and MERRA2 data for a specific location and year.
    """
    retrieve_status = True
    data_noaa, tz, distance, elevation, wmo, station_name, state, country, epw_exists = get_data_noaa(lat, lon, year, save_folder)

    if epw_exists:
        return '', '', '', '', '', '', wmo, epw_exists

    info_dict = {
        'timeshift': get_time_shift(tz),
        'elevation': elevation,
        'wmo': wmo,
        'station_name': station_name,
        'state': state,
        'country': country,
        'lat': lat,
        'lon': lon,
        'weather_file_type': file_type
    }

    try:
        data_noaa_tz_adj = filter_dataframe_by_date(convert_utc_to_local(data_noaa, tz), datetime(year, 1, 1), datetime(year+1, 1, 1))
    except AttributeError:
        print("We don't have NOAA data for this location/year")
        return [], False, info_dict, np.nan, '', '', wmo, False

    missing_hours_num, largest_consecutive_group = check_missing_hours(year, data_noaa_tz_adj)
    if largest_consecutive_group > 3:
        print('More than 3 consecutive missing hours')
        return [], False, info_dict, np.nan, '', '', wmo, False

    data_noaa_tz_adj_h = data_noaa_tz_adj.resample('H').mean()
    data_noaa_tz_adj_h_interpolated = data_noaa_tz_adj_h.interpolate(method='linear', limit=3, limit_direction='forward')
    hdd, cdd = calculate_hdd_cdd(data_noaa_tz_adj_h_interpolated, 'temp')

    df_merra2, header_merra2 = get_parameters_MERRA2(lat, lon, year)
    df_merged = merge_data(df_merra2, data_noaa_tz_adj_h_interpolated)

    return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, epw_exists


def run_individual_location(lat, lon, year, file_type, save_folder):
    """
    Processes a single location, fetching data and handling errors.
    """
    data_meteostat_merra2, retrieve_status, info_dict, distance, hdd, cdd, wmo, epw_exists = get_noaa_merra2_data(lat, lon, year, file_type, save_folder)

    if epw_exists:
        return True, '', wmo, '', '', True

    if retrieve_status:
        output_path = os.path.join(save_folder, f"{info_dict['wmo']}_{year}.epw")
        data_meteostat_merra2.to_csv(output_path, header=False, index=False)
        with open(output_path, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_path, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)

    return retrieve_status, distance, info_dict['wmo'], hdd, cdd, False


def get_time_shift(timezone_name):
    """
    Calculates the time shift for a given timezone from UTC.
    """
    timezone = pytz.timezone(timezone_name)
    now = datetime.now(timezone)
    utc_offset = now.utcoffset()
    return int(utc_offset.total_seconds() // 3600)  # Return hours offset only


def calculate_hdd_cdd(df, temperature_column):
    """
    Calculate Heating Degree Days (HDD) and Cooling Degree Days (CDD) from hourly temperature data in Celsius.
    """
    df[temperature_column] = df[temperature_column] * 9 / 5 + 32
    base_temperature = 65

    df['date'] = df.index.to_series().dt.date
    daily_mean_temp = df.groupby('date')[temperature_column].mean().reset_index()
    daily_mean_temp.columns = ['date', 'mean_temp']

    daily_mean_temp['HDD'] = (base_temperature - daily_mean_temp['mean_temp']).clip(lower=0)
    daily_mean_temp['CDD'] = (daily_mean_temp['mean_temp'] - base_temperature).clip(lower=0)

    total_hdd = daily_mean_temp['HDD'].sum()
    total_cdd = daily_mean_temp['CDD'].sum()

    return int(total_hdd), int(total_cdd)


def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)
    epw_exists = False
    station_number = 0
    len_data = 0

    while len_data < 8000:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[0]))
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)

    if epw_exists:
        return '', '', '', '', wmo, '', '', '', epw_exists

    station_info = stations.fetch(station_number)
    timezone = station_info['timezone'].values[0]
    elevation = station_info['elevation'].values[0]
    distance = stations.fetch()['distance'].values[0]
    wmo = fix_wmo(str(station_info.index.values[0]))
    station_name = station_info['name'].values[0]
    state = station_info['region'].values[0]
    country = station_info['country'].values[0]

    return data, timezone, distance, elevation, wmo, station_name, state, country, epw_exists


# Helper function to check and update missing data
def update_if_missing(df, index, col_name, new_value):
    if pd.isna(df.at[index, col_name]) or not df.at[index, col_name]:
        df.at[index, col_name] = new_value


def retrieve_info_other_location(wmo, zipcodes, year):
    retrieve_status = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"Do we have data for {year}?"].values[0]
    distance = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"distance_location_station_miles_{year}"].values[0]
    hdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"hdd_base65F_{year}"].values[0]
    cdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"cdd_base65F_{year}"].values[0]
    return retrieve_status, distance, hdd, cdd


# Define constants
year = 2022
file_type = 'AMY'
save_folder = 'epws_wmo'

# Load the zip codes CSV once
zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# Process each row in the DataFrame starting from the specified index
for index, row in zipcodes.iloc[1400:].iterrows():
    print(index)
    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed
    lat = row['lat']
    lon = row['lng']

    # Retrieve data for the current location
    retrieve_status, distance, wmo, hdd, cdd, retrieve_info_closest_other_locations = run_individual_location(lat, lon, year, file_type, save_folder)

    if retrieve_info_closest_other_locations:
        retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)

    # Update the DataFrame only if the cell is empty or contains placeholder (like 'nan')
    update_if_missing(zipcodes, index, f"Do we have data for {year}?", retrieve_status)
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance * 0.000621371)  # Convert from meters to miles
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)

# # Save the DataFrame to the CSV file after all iterations are complete
# zipcodes.to_csv('resources/zip_code_list.csv', index=False)


    # Save the DataFrame to the CSV file after each iteration
    zipcodes.to_csv('resources/zip_code_list.csv', index=False)

    # Reopen the file to ensure the latest version is loaded
    zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})


1400
1401
1402
1403
1404
1405
1406
1407
1408
1409
More than 3 consecutive missing hours
1410
1411
1412
1413
1414
1415
1416
1417
1418
More than 3 consecutive missing hours
1419
1420
1421
1422
1423
1424
1425
1426
1427
1428
1429
1430
1431
1432
1433
1434
1435
1436
1437
1438
1439
1440
1441
1442
1443
1444
1445
More than 3 consecutive missing hours
1446
1447
1448
1449
1450
1451
1452
1453
1454
1455
1456
1457
1458
1459
1460
1461
1462
1463
1464
1465
1466
1467
1468
1469
1470
1471


1472
1473
1474
1475
1476
1477
1478
1479
1480
1481
1482
1483
1484
1485
1486
1487
1488
1489
1490
1491
1492
1493
1494
1495
1496
1497
1498
1499
1500
1501
1502
1503
1504
1505
1506
1507
1508
1509
1510
1511
1512
1513
1514
1515
1516
1517
1518
1519
1520
1521
1522
1523
1524
1525
1526
1527
1528
1529
1530
1531
1532
1533
1534
1535
1536
1537
1538
1539
1540
1541


1542
1543
1544
1545
1546
1547
1548
1549
1550
1551


1552
1553
1554


KeyboardInterrupt: 

In [66]:
get_wmo_from_icao_NOAA('KIJD0')

In [54]:
lat = 51.97796
lon = -130.03671
year = 2023

def get_data_noaa(lat, lon, year):
    #  The time zone used by Meteostat is Coordinated Universal Time (UTC).
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year-1, 12, 31)
    end = datetime(year+1, 1, 2)

    # Use certifi's CA bundle
    # ssl_context = ssl.create_default_context(cafile=certifi.where())
    # Find the closest station
    stations = Stations().nearby(lat, lon) 
    # Get hourly data for the first station
    station_number = 0
    len_data = 0
    while len_data < 8000:
        station_number += 1
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)

    print('final_station')
    print(station_number)
    timezone = stations.fetch(station_number)['timezone'].values[0]
    #elevation in meters
    elevation = stations.fetch(station_number)['elevation'].values[0]
    #Distance in m
    distance = stations.fetch()['distance'].values[0]
    print(distance)
    #WMO
    wmo = str(stations.fetch(station_number).index.values[0])
    #Station Name
    station_name = stations.fetch()['name'].values[0]
    #State
    state = stations.fetch()['region'].values[0]
    #Country
    country = stations.fetch()['country'].values[0]

    return data, timezone, distance, elevation, wmo, station_name, state, country

data, timezone, distance, elevation, wmo, station_name, state, country= get_data_noaa(lat, lon, year)

final_station
1
67507.72686757684


In [21]:
data['prcp'].mean()

0.28986264048132593

In [ ]:

file_type = 'AMY'
lat = 41.766595
lon = -88.318735
year = 2023
name= 'TEST_2023'
output_name = name + '.epw'

# Run your existing code with these parameters
data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
data_meteostat_merra2.to_csv(output_name, header=False, index=False)
with open(output_name, 'r') as original_file:
    data_content = original_file.read()
header_lines = create_header(data_meteostat_merra2, year, info_dict)
with open(output_name, 'w') as new_file:
    new_file.write("\n".join(header_lines) + "\n" + data_content)


In [ ]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QLineEdit, QPushButton, QVBoxLayout, QGridLayout, QMessageBox
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io

# Assuming all the previous functions are defined above or imported from another module


def run_individual_location(output_name, lat, lon, year, file_type, name):
    try:
        # Run your existing code with these parameters
        data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
        data_meteostat_merra2.to_csv(output_name, header=False, index=False)
        with open(output_name, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_name, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
        QMessageBox.information(window, "Success", f"Data saved successfully to {output_name}")
    except Exception as e:
        QMessageBox.critical(window, "Error", f"An error occurred: {str(e)}")


def on_run_clicked():
    lat = float(lat_input.text())
    lon = float(lon_input.text())
    year = int(year_input.text())
    file_type = file_type_input.text()
    name = name_input.text()
    output_name = output_name_input.text()
    
    run_individual_location(output_name, lat, lon, year, file_type, name)


# Initialize the application
app = QApplication(sys.argv)

# Create the main window
window = QWidget()
window.setWindowTitle("NOAA MERRA2 Data Processor")
window.setGeometry(100, 100, 400, 300)

# Create a grid layout
layout = QGridLayout()

# Add widgets for input fields
layout.addWidget(QLabel("Latitude:"), 0, 0)
lat_input = QLineEdit()
layout.addWidget(lat_input, 0, 1)
lat_input.setText("41.766595")

layout.addWidget(QLabel("Longitude:"), 1, 0)
lon_input = QLineEdit()
layout.addWidget(lon_input, 1, 1)
lon_input.setText("-88.318735")

layout.addWidget(QLabel("Year:"), 2, 0)
year_input = QLineEdit()
layout.addWidget(year_input, 2, 1)
year_input.setText("2023")

layout.addWidget(QLabel("File Type:"), 3, 0)
file_type_input = QLineEdit()
layout.addWidget(file_type_input, 3, 1)
file_type_input.setText("AMY")

layout.addWidget(QLabel("Name:"), 4, 0)
name_input = QLineEdit()
layout.addWidget(name_input, 4, 1)
name_input.setText("TEST_2023")

layout.addWidget(QLabel("Output File Name:"), 5, 0)
output_name_input = QLineEdit()
layout.addWidget(output_name_input, 5, 1)
output_name_input.setText("TEST_2023.epw")

# Add a run button
run_button = QPushButton("Run")
run_button.clicked.connect(on_run_clicked)
layout.addWidget(run_button, 6, 0, 1, 2)

# Set the layout for the main window
window.setLayout(layout)

# Show the window
window.show()

# Run the application's main loop
sys.exit(app.exec_())


## Figure out Zip Codes

In [ ]:
import pandas as pd
import numpy as np

# Define a function to calculate the Haversine distance between two points in km
def haversine(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula to calculate the distance
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

# 1) Open the file resources/zip_code_list.csv as a dataframe and call it zipcodes
zipcodes = pd.read_csv('resources/zip_codes_list.csv')

# 2) Open design_conditions.csv as a dataframe and call it dc
dc = pd.read_csv('resources/meteostat_stats.csv')

# Ensure latitude and longitude columns are in float format
zipcodes['lat'] = zipcodes['lat'].astype(float)
zipcodes['lng'] = zipcodes['lng'].astype(float)
dc['latitude'] = dc['latitude'].astype(float)
dc['longitude'] = dc['longitude'].astype(float)

# 3) Initialize columns in zipcodes dataframe for storing results
zipcodes['Distance'] = np.nan
zipcodes['Location'] = ""

# 4) Loop through all the rows in zipcodes
for idx, row in zipcodes.iterrows():
    lat1 = row['lat']
    lon1 = row['lng']

    # Calculate the distance to each location in the dc dataframe
    dc['Distance'] = dc.apply(lambda x: haversine(lat1, lon1, x['latitude'], x['longitude']), axis=1)

    # 5) Find the closest location in dc
    closest_location = dc.loc[dc['Distance'].idxmin()]

    # 6) Update the zipcodes dataframe with the closest location's details
    zipcodes.at[idx, 'Distance'] = closest_location['Distance']
    zipcodes.at[idx, 'Location'] = closest_location['name']

# Display the updated dataframe
zipcodes.to_csv('resources/updated_zip_code_list_again.csv', index=False)

In [ ]:
# Ensure the 'zip' column is a string and pad with zeros to make it 5 digits
zipcodes['zip0'] = zipcodes['zip'].astype(str).str.zfill(5)

zipcodes


In [ ]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

In [ ]:
import pandas as pd
import requests
from io import StringIO

# URL of the dataset containing ZIP codes and their respective coordinates
url = "https://raw.githubusercontent.com/scpike/us-state-county-zip/master/geo-data.csv"

# Fetching the CSV file from the URL
response = requests.get(url)
response.raise_for_status()  # Raises an error for bad responses

# Reading the CSV data into a pandas DataFrame
data = pd.read_csv(StringIO(response.text))


data.to_csv('resources/zip_codes_list.csv')


In [ ]:
import pandas as pd
import requests
from io import BytesIO
from zipfile import ZipFile

# URL of a dataset containing ZIP codes, latitude, and longitude
url = "https://simplemaps.com/static/data/us-zips/1.74/basic/simplemaps_uszips_basicv1.74.zip"

# Download the ZIP file with SSL verification disabled
response = requests.get(url, verify=False)  # Bypass SSL certificate verification
response.raise_for_status()  # Check if the request was successful

# Unzip the file and read the CSV
with ZipFile(BytesIO(response.content)) as zip_file:
    # Extract the CSV file within the ZIP
    with zip_file.open('uszips.csv') as file:
        zip_code_data = pd.read_csv(file)

# # Display the DataFrame to verify the contents
# print(zip_code_data.head())

# # Save the DataFrame to a local CSV file
zip_code_data.to_csv('resources/zip_codes_list.csv', index=False)
# print("Data saved to 'us_zip_codes_with_coordinates.csv'.")


In [ ]:
zip_code_data